# Variational imaginary-time MaxCut

All 22 source stages, a fresh four-node weighted graph, six ordered generators, fourteen imaginary-time steps and 128 shots per histogram. Report update energy, independently measured energy, rounded cut and the exact four-node comparison.

The source graph generator has n=k=4, so the native workflow constructs the complete four-node graph and draws the weights from seed zero. It recomputes the sorted generators from that fresh graph and specializes the six angles before each local execution. A separate local shot seed replaces hosted randomness. The rounded partition keeps its actual measured cost.

## Load the native workflow

In [ ]:
from pathlib import Path

from IPython.display import Image, display

from examples.synthesis.applications.optimization.variational_quantum_imaginary_time_evolution.overview.model import (
    source_model,
)
from examples.synthesis.applications.optimization.variational_quantum_imaginary_time_evolution.overview.plots import (
    save_graph,
    save_trace,
)
from examples.synthesis.applications.optimization.variational_quantum_imaginary_time_evolution.overview.quantum import (
    probabilities,
    program_builder,
)
from examples.synthesis.applications.optimization.variational_quantum_imaginary_time_evolution.overview.reference import (
    direct_cut,
)
from examples.synthesis.applications.optimization.variational_quantum_imaginary_time_evolution.overview.workflow import (
    ROUNDS,
    SHOTS,
    run_workflow,
)

folder = Path("artifacts/development/imaginary-time-maxcut")


## Choose how to display the weighted graph

In [ ]:
graph_figure = folder / "graph.png"


## Set the MaxCut convention

In [ ]:
print("The objective is the negative cut weight.")


## Regenerate the source graph

In [ ]:
model = source_model()


## Keep all four nodes

In [ ]:
assert len(model.edges) == 6


## Inspect the six fresh edges

In [ ]:
print("Graph edges:", model.edges)


## Display the graph

In [ ]:
display(Image(filename=str(save_graph(model, graph_figure))))


## Check the local degrees

In [ ]:
print(
    "Weighted degrees:",
    [sum(w for u, v, w in model.edges if i in (u, v)) for i in range(4)],
)


## Order the pair rotations

In [ ]:
print("Z and Y nodes:", model.generators)


## Build the checked ansatz

In [ ]:
build = program_builder(model)


## Retain all six angles

In [ ]:
initial = (0.0,) * len(model.generators)


## Check the generator count

In [ ]:
assert len(initial) == 6


## Synthesize at the zero start

In [ ]:
program = build(initial)


## Read the exact prepared distribution

In [ ]:
print("Initial named probabilities:", probabilities(program))


## Retain the constant energy offset

In [ ]:
print("The Hamiltonian includes the identity term of the cut cost.")


## Inspect the complete sixteen-word cost table

In [ ]:
print("Hamiltonian diagonal:", model.energy.tolist())


## Set the finite sampling and shift rule

In [ ]:
print("Shots per histogram and rounds:", SHOTS, ROUNDS)


## Record the partition readout rule

In [ ]:
print("Round each local spin using its independently sampled sign.")


## Set the starting point

In [ ]:
print("Initial angles:", initial)


## Execute all fourteen updates

In [ ]:
result = run_workflow()
print(
    "Number of histogram evaluations and shots:",
    result.measurements,
    result.measurements * SHOTS,
)
for row in result.rounds:
    print(
        f"Iteration {row.iteration}:",
        row.energy,
        row.measured_energy,
        row.partition_word,
        row.partition_energy,
    )


## Enumerate the four-node reference

In [ ]:
print(
    "Exact positive maximum cut:",
    max(direct_cut(result.model, word) for word in range(16)),
)


## Compare measured cut and energy

In [ ]:
display(Image(filename=str(save_trace(result, folder / "trace.png"))))
